### Initialization

In [1]:
import pandas as pd
from datasets import load_dataset
from torchao.quantization import (
    Int8DynamicActivationInt8WeightConfig,
    Int8WeightOnlyConfig,
)
import torch

from sequence_tagging_benchmark.taggers import (
    NltkHMMTagger,
    HmmlearnTagger,
    CRFTagger,
    BiLSTMTagger,
    TransformerTagger,
    QuantizedBiLSTMTagger,
    QuantizedTransformerTagger,
    ONNXTransformerTagger,
)

/Users/nikita/anaconda3/envs/nlp_case_study/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0315 11:05:06.112000 69024 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
NUM_THREADS = 8

torch.set_num_threads(NUM_THREADS)

### Data loading

In [3]:
dataset = load_dataset("tner/ontonotes5")

train_data = dataset["train"]
test_data = dataset["test"]

# Load manually from HuggingFace
id2label = {
    0: "O",
    1: "B-CARDINAL",
    2: "B-DATE",
    3: "I-DATE",
    4: "B-PERSON",
    5: "I-PERSON",
    6: "B-NORP",
    7: "B-GPE",
    8: "I-GPE",
    9: "B-LAW",
    10: "I-LAW",
    11: "B-ORG",
    12: "I-ORG",
    13: "B-PERCENT",
    14: "I-PERCENT",
    15: "B-ORDINAL",
    16: "B-MONEY",
    17: "I-MONEY",
    18: "B-WORK_OF_ART",
    19: "I-WORK_OF_ART",
    20: "B-FAC",
    21: "B-TIME",
    22: "I-CARDINAL",
    23: "B-LOC",
    24: "B-QUANTITY",
    25: "I-QUANTITY",
    26: "I-NORP",
    27: "I-LOC",
    28: "B-PRODUCT",
    29: "I-TIME",
    30: "B-EVENT",
    31: "I-EVENT",
    32: "I-FAC",
    33: "B-LANGUAGE",
    34: "I-PRODUCT",
    35: "I-ORDINAL",
    36: "I-LANGUAGE",
}

In [4]:
raw_train_tokens = [row["tokens"] for row in train_data]
raw_train_tags = [[id2label[tag_id] for tag_id in row["tags"]] for row in train_data]

raw_test_tokens = [row["tokens"] for row in test_data]
raw_test_tags = [[id2label[tag_id] for tag_id in row["tags"]] for row in test_data]

### Benchmarking

In [ ]:
TAG_TYPE = "ner"
BLSTM_MODEL_NAME = "flair/ner-english-ontonotes-fast"
TRANSFORMER_MODEL_NAME = "pitangent-ds/roberta-base-ontonotes"

nltk_hmm = NltkHMMTagger()
nltk_hmm.train(raw_train_tokens, raw_train_tags)

hmmlearn_hmm = HmmlearnTagger()
hmmlearn_hmm.train(raw_train_tokens, raw_train_tags)

crf = CRFTagger()
crf.train(raw_train_tokens, raw_train_tags)

bilstm = BiLSTMTagger(model_name=BLSTM_MODEL_NAME, tag_type=TAG_TYPE)
bilstm.train(raw_train_tokens, raw_train_tags)

dynamic_quantized_bilstm = QuantizedBiLSTMTagger(
    model_name=BLSTM_MODEL_NAME,
    tag_type=TAG_TYPE,
    quantize_config=Int8DynamicActivationInt8WeightConfig(),
)
dynamic_quantized_bilstm.train(raw_train_tokens, raw_train_tags)

weight_only_quantized_bilstm = QuantizedBiLSTMTagger(
    model_name=BLSTM_MODEL_NAME,
    tag_type=TAG_TYPE,
    quantize_config=Int8WeightOnlyConfig(),
)
weight_only_quantized_bilstm.train(raw_train_tokens, raw_train_tags)

transformer = TransformerTagger(
    model_name=TRANSFORMER_MODEL_NAME,
    id2label=id2label,
)
transformer.train(raw_train_tokens, raw_train_tags)

dynamic_quantized_transformer = QuantizedTransformerTagger(
    model_name=TRANSFORMER_MODEL_NAME,
    id2label=id2label,
    quantize_config=Int8DynamicActivationInt8WeightConfig(),
)
dynamic_quantized_transformer.train(raw_train_tokens, raw_train_tags)

weight_only_quantized_transformer = QuantizedTransformerTagger(
    model_name=TRANSFORMER_MODEL_NAME,
    id2label=id2label,
    quantize_config=Int8WeightOnlyConfig(),
)
weight_only_quantized_transformer.train(raw_train_tokens, raw_train_tags)

onnx_transformer = ONNXTransformerTagger(
    model_name=TRANSFORMER_MODEL_NAME,
    id2label=id2label,
)
onnx_transformer.train(raw_train_tokens, raw_train_tags)

In [ ]:
# Evaluate all models
results = []

# Traditional models
results.append(
    {
        "name": "nltk_hmm",
        **nltk_hmm.evaluate(raw_test_tokens, raw_test_tags, batch_size=1),
    }
)
results.append(
    {
        "name": "hmmlearn_hmm",
        **hmmlearn_hmm.evaluate(raw_test_tokens, raw_test_tags, batch_size=1),
    }
)
results.append(
    {"name": "crf", **crf.evaluate(raw_test_tokens, raw_test_tags, batch_size=1)}
)

# BiLSTM variants
results.append(
    {
        "name": "bilstm-batch-1",
        **bilstm.evaluate(raw_test_tokens, raw_test_tags, batch_size=1),
    }
)
results.append(
    {
        "name": "bilstm-batch-128",
        **bilstm.evaluate(raw_test_tokens, raw_test_tags, batch_size=128),
    }
)
results.append(
    {
        "name": "bilstm-batch-128-smart-batching",
        **bilstm.evaluate(
            raw_test_tokens, raw_test_tags, batch_size=128, use_smart_batching=True
        ),
    }
)
results.append(
    {
        "name": "dynamic-quantized-bilstm-batch-128",
        **dynamic_quantized_bilstm.evaluate(
            raw_test_tokens, raw_test_tags, batch_size=128
        ),
    }
)
results.append(
    {
        "name": "weight-only-quantized-bilstm-batch-128",
        **weight_only_quantized_bilstm.evaluate(
            raw_test_tokens, raw_test_tags, batch_size=128
        ),
    }
)

# Transformer variants
results.append(
    {
        "name": "transformer-batch-1",
        **transformer.evaluate(raw_test_tokens, raw_test_tags, batch_size=1),
    }
)
results.append(
    {
        "name": "transformer-batch-128",
        **transformer.evaluate(raw_test_tokens, raw_test_tags, batch_size=128),
    }
)
results.append(
    {
        "name": "transformer-batch-128-smart-batching",
        **transformer.evaluate(
            raw_test_tokens, raw_test_tags, batch_size=128, use_smart_batching=True
        ),
    }
)
results.append(
    {
        "name": "dynamic-quantized-transformer-batch-128",
        **dynamic_quantized_transformer.evaluate(
            raw_test_tokens, raw_test_tags, batch_size=128
        ),
    }
)
results.append(
    {
        "name": "weight-only-quantized-transformer-batch-128",
        **weight_only_quantized_transformer.evaluate(
            raw_test_tokens, raw_test_tags, batch_size=128
        ),
    }
)
results.append(
    {
        "name": "onnx-transformer-batch-128",
        **onnx_transformer.evaluate(raw_test_tokens, raw_test_tags, batch_size=128),
    }
)
results = pd.DataFrame(results)

In [ ]:
results.to_csv("../artifacts/ner_results.csv")

In [ ]:
results